# 13_Zip_Complete_Project_For_Team.ipynb

## Purpose
Creates a ZIP of `/home/sagemaker-user/Heart_Attack_Risk_Assessment` for team sharing, verifies it, and displays a download link.

It excludes common caches, notebook checkpoints, virtual environments, and common credential files.

**Security:** Before sharing, check that AWS keys, passwords, tokens, or other secrets were not manually pasted into notebooks, scripts, JSON, or configuration files.


## 1. Locate project and configure ZIP

In [10]:
from pathlib import Path
from datetime import datetime
import zipfile

PROJECT_ROOT = Path("/home/sagemaker-user/Heart_Attack_Risk_Assessment")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_ROOT}")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
ZIP_FILE = PROJECT_ROOT.parent / f"Heart_Attack_Risk_Assessment_Team05_{timestamp}.zip"

print("Project :", PROJECT_ROOT)
print("ZIP     :", ZIP_FILE)


## 2. Define safe exclusions

In [11]:
EXCLUDE_DIRS = {
    ".git", ".ipynb_checkpoints", "__pycache__",
    ".pytest_cache", ".mypy_cache", ".venv", "venv", "env",
}

EXCLUDE_FILES = {
    ".DS_Store", "Thumbs.db", ".env", "credentials",
}

EXCLUDE_SUFFIXES = {".pyc", ".pyo"}

def should_exclude(path: Path):
    relative = path.relative_to(PROJECT_ROOT)

    if any(part in EXCLUDE_DIRS for part in relative.parts):
        return True

    if path.name in EXCLUDE_FILES:
        return True

    if path.suffix in EXCLUDE_SUFFIXES:
        return True

    return False

print("✅ Exclusion rules ready.")


## 3. Create the project ZIP

In [12]:
file_count = 0
total_bytes = 0

print("=" * 80)
print("CREATING TEAM PROJECT ZIP")
print("=" * 80)

with zipfile.ZipFile(
    ZIP_FILE,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zipf:

    for path in PROJECT_ROOT.rglob("*"):
        if not path.is_file() or should_exclude(path):
            continue

        archive_name = Path(PROJECT_ROOT.name) / path.relative_to(PROJECT_ROOT)
        zipf.write(path, arcname=archive_name)

        file_count += 1
        total_bytes += path.stat().st_size

print("✅ ZIP creation completed.")


## 4. Verify ZIP and print summary

In [13]:
with zipfile.ZipFile(ZIP_FILE, "r") as zipf:
    bad_file = zipf.testzip()

if bad_file is not None:
    raise RuntimeError(f"ZIP verification failed at: {bad_file}")

print("=" * 80)
print("ZIP CREATED SUCCESSFULLY")
print("=" * 80)
print("Files included :", file_count)
print("Original size :", round(total_bytes / (1024 ** 2), 2), "MB")
print("ZIP size      :", round(ZIP_FILE.stat().st_size / (1024 ** 2), 2), "MB")
print("\nZIP location:")
print(ZIP_FILE)
print("\n✅ ZIP integrity check passed.")
print("✅ Project is ready to download.")


## 5. Download the ZIP

In [14]:
from IPython.display import display, FileLink

print("=" * 80)
print("DOWNLOAD PROJECT")
print("=" * 80)

display(
    FileLink(
        str(ZIP_FILE),
        result_html_prefix="<b>Click here to download the complete project:</b><br>",
    )
)


## 6. Optional — Inspect ZIP contents

Run this only if you want to check exactly which files are included before sharing.


In [15]:
with zipfile.ZipFile(ZIP_FILE, "r") as zipf:
    names = zipf.namelist()

print("Total files:", len(names))
print("=" * 80)

for name in names:
    print(name)


## Recommended use

Run Sections **1 → 5** in order. Section 6 is optional.

The ZIP is created outside the project folder, so the ZIP cannot accidentally include itself.
